# Прогнозирование цен на недвижимость г. Омск

---

**Структура ноутбука:**

| Этап | Описание |
|------|---------|
| 0 | Импорты и конфигурация |
| 1 | Загрузка, нормализация, создание адреса |
| 2 | Геокодирование и POI-признаки |
| 3 | Детерминированные признаки + фильтр цены |
| 4 | Разделение train / test (до импутации) |
| 5 | Импутация пропусков (только по train) |
| 6 | Выбросы и нормализация (только по train) |
| 7 | Подготовка признаков: OHE / Target Encoding / B2 |
| 8 | Обучение моделей, подбор гиперпараметров, метрики |
| 9 | Сохранение модели и предсказание |

**Ключевые решения:**

1. Train/test split перенесён до импутации и нормализации — утечка данных устранена
2. Все статистики (мода, медианы, коэффициенты, границы winsorizing) считаются только по train
3. Добавлены флаги импутации: `living_imputed`, `kitchen_imputed`, `year_imputed`
4. Добавлены признаки этажности: `floor_ratio`, `is_first_floor`, `is_last_floor`
5. Обучение на log(price), предсказание через exp()
6. Гиперпараметры подбираются отдельно для каждой конфигурации признаков
7. Подход B2 (без living/kitchen) — для оценки влияния мультиколлинеарности
8. Выбор лучшей модели по CV MAE, добавлены Median AE и анализ по сегментам

> **Как запустить:** загрузите файлы `all.csv`, `coordinates.csv`,
> `2гис_еда.csv`, `2гис_образование.csv`, `2гис_культура.csv`, `2гис_здоровье.csv`
> в `/content/`, затем запустите все ячейки последовательно.


## Этап 0. Импорты и конфигурация

In [1]:
!pip install xgboost shap

import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from math import radians, cos, sin, asin, sqrt
from typing import Dict, List

from scipy.spatial.distance import cdist
from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, median_absolute_error, make_scorer)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from xgboost import XGBRegressor
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA_DIR   = '/content/'
OUTPUT_DIR = '/content/output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CENTER_LAT = 54.991227
CENTER_LON = 73.365562

MISSING_THRESHOLD = 30.0
POI_RADII = [500, 1000]

print('Импорты выполнены')
print(f'Выходные файлы: {OUTPUT_DIR}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 1.2 MB/s eta 0:00:00
Импорты выполнены
Выходные файлы: /content/output/


## Этап 1. Загрузка, нормализация и создание адреса

### 1.1 Вспомогательные функции


In [2]:
def count_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for col in df.columns:
        series = df[col]
        n_null = series.isnull().sum()
        if pd.api.types.is_numeric_dtype(series):
            n_sentinel = (series == -1).sum()
        else:
            n_sentinel = series.astype(str).str.strip().isin(['-1', '']).sum()
        missing = n_null + n_sentinel
        stats.append({
            'column': col,
            'missing_count': missing,
            'missing_pct': round(100 * missing / len(df), 2),
            'total_rows': len(df)
        })
    return pd.DataFrame(stats).sort_values('missing_pct', ascending=False)

print('Функции определены')


Функции определены


### 1.2 Загрузка и очистка

In [3]:
df_raw = pd.read_csv(DATA_DIR + 'all.csv', delimiter=';')
print(f'Исходный размер: {df_raw.shape}')

DROP_ALWAYS = ['author', 'author_type', 'phone', 'accommodation_type',
               'deal_type', 'location', 'underground']
df = df_raw.drop(columns=[c for c in DROP_ALWAYS if c in df_raw.columns])

missing = count_missing_values(df)
cols_to_drop = missing[missing['missing_pct'] > MISSING_THRESHOLD]['column'].tolist()
print(f'Удаляем столбцы (>{MISSING_THRESHOLD}% пропусков): {cols_to_drop}')
df = df.drop(columns=cols_to_drop)

before = len(df)
no_addr = (df['house_number'].isna() |
           (df['house_number'].astype(str).str.strip() == ''))
df = df[~no_addr].reset_index(drop=True)
print(f'Удалено объявлений без адреса: {before - len(df)}')
print(f'Итоговый размер: {df.shape}')


Исходный размер: (7604, 24)
Удаляем столбцы (>30.0% пропусков): ['object_type', 'heating_type', 'house_material_type', 'finish_type', 'residential_complex']
Удалено объявлений без адреса: 461
Итоговый размер: (7143, 12)


### 1.3 Приведение типов и создание адреса

Тип-кастинг и обработка сентинелей — детерминированные преобразования,
их безопасно применять до сплита.

Адрес формируется здесь без `.str.strip()` — чтобы сохранить пробелы,
с которыми были геокодированы адреса в `coordinates.csv`.


In [4]:
df_typed = df.copy()

df_typed['living_meters'] = (
    df_typed['living_meters']
    .replace('-1', pd.NA)
    .astype(str).str.replace(',', '.', regex=False)
    .str.extract(r'([\d\.]+)')[0]
    .astype(float)
)

df_typed['kitchen_meters'] = (
    df_typed['kitchen_meters']
    .replace('-1', pd.NA)
    .astype(str).str.replace(',', '.', regex=False)
    .str.extract(r'([\d\.]+)')[0]
    .astype(float)
)

df_typed['year_of_construction'] = (
    df_typed['year_of_construction']
    .replace('-1', pd.NA)
    .astype(str)
    .str.extract(r'(\b\d{4}\b)', expand=False)
    .astype(float)
)

df_typed['rooms_count'] = (
    df_typed['rooms_count']
    .replace(['-1', -1], np.nan)
    .astype(float)
)

# Адрес — без strip(), чтобы совпадать с coordinates.csv
df_typed['address'] = (
    'Омская область, Омск, ' +
    df_typed['street'].astype(str) + ', ' +
    df_typed['house_number'].astype(str)
)
df_typed['address'] = df_typed['address'].str.replace('  ', ' ')

print('Типы приведены, адрес создан')
print('\nПропущенные значения:')
print(count_missing_values(df_typed).query('missing_count > 0')
      [['column','missing_count','missing_pct']].to_string(index=False))


Типы приведены, адрес создан

Пропущенные значения:
              column  missing_count  missing_pct
       living_meters           1662        23.27
year_of_construction           1056        14.78
      kitchen_meters            536         7.50
         rooms_count             31         0.43


## Этап 2. Геокодирование и POI-признаки

Геокодирование и расчёт POI — детерминированные join-операции на внешних данных.
Их безопасно выполнять до сплита.


In [5]:
coords_df = pd.read_csv(DATA_DIR + 'coordinates.csv', encoding='utf-8')
if 'Address' in coords_df.columns:
    coords_df = coords_df.rename(columns={'Address': 'address'})

sample_lat = coords_df['Latitude'].iloc[0]
sample_lon = coords_df['Longitude'].iloc[0]
if 72 < sample_lat < 75 and 54 < sample_lon < 56:
    coords_df = coords_df.rename(columns={'Latitude': 'Longitude', 'Longitude': 'Latitude'})
    print('Координаты были перепутаны — исправлено')
else:
    print('Порядок координат корректный')

df_geo = df_typed.merge(
    coords_df[['address', 'Latitude', 'Longitude']],
    on='address', how='left'
)

missing_coords = df_geo['Latitude'].isna().sum()
print(f'Объявлений без координат: {missing_coords} ({100*missing_coords/len(df_geo):.1f}%)')
df_geo = df_geo.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
print(f'После удаления: {df_geo.shape}')


Порядок координат корректный
Объявлений без координат: 0 (0.0%)
После удаления: (7143, 15)


### 2.1 Загрузка данных 2ГИС

In [6]:
class GISDataProcessor:
    CATEGORY_MAPPING = {
        'еда':        ['Кафе', 'Рестораны'],
        'здоровье':   ['Многопрофильные медицинские центры', 'Больницы'],
        'культура':   ['Кинотеатры', 'Музеи', 'Театры'],
        'образование':['Гимназии', 'Лицеи', 'Школы', 'Детские сады',
                       'Университеты', 'Колледжи']
    }
    RUBRIC_TO_SUBTYPE = {
        'Кафе': 'кафе', 'Рестораны': 'рестораны',
        'Многопрофильные медицинские центры': 'медцентры', 'Больницы': 'больницы',
        'Кинотеатры': 'кинотеатры', 'Музеи': 'музеи', 'Театры': 'театры',
        'Гимназии': 'гимназии', 'Лицеи': 'лицеи', 'Школы': 'школы',
        'Детские сады': 'детские_сады', 'Университеты': 'университеты',
        'Колледжи': 'колледжи'
    }
    OMSK_BOUNDS = {'lat_min': 54.5, 'lat_max': 55.5, 'lon_min': 72.5, 'lon_max': 74.5}

    def load_file(self, path, category):
        df = pd.read_csv(path, sep=';', encoding='utf-8')
        required = ['Наименование', 'Рубрики', 'Широта', 'Долгота']
        if not all(c in df.columns for c in required):
            return pd.DataFrame()
        for col in ['Широта', 'Долгота']:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.strip().str.replace(',', '.'), errors='coerce')
        df = df.dropna(subset=['Широта', 'Долгота'])
        b = self.OMSK_BOUNDS
        df = df[df['Широта'].between(b['lat_min'], b['lat_max']) &
                df['Долгота'].between(b['lon_min'], b['lon_max'])]
        target_rubrics = self.CATEGORY_MAPPING.get(category, [])
        df = df[df['Рубрики'].fillna('').apply(
            lambda s: any(r in s for r in target_rubrics))]
        df['category'] = category
        df['subtype'] = df['Рубрики'].fillna('').apply(
            lambda s: next((self.RUBRIC_TO_SUBTYPE[r]
                            for r in self.RUBRIC_TO_SUBTYPE if r in s), 'other'))
        return df[['Наименование', 'Широта', 'Долгота', 'category', 'subtype']]

    def load_all(self, file_map):
        frames = []
        for cat, path in file_map.items():
            if os.path.exists(path):
                print(f'  Загружаем: {cat}...')
                df = self.load_file(path, cat)
                print(f'    → {len(df)} объектов')
                frames.append(df)
        result = pd.concat(frames, ignore_index=True)
        print(f'\nИтого POI: {len(result)}')
        return result

processor = GISDataProcessor()
file_map = {
    'еда':        DATA_DIR + '2гис_еда.csv',
    'здоровье':   DATA_DIR + '2гис_здоровье.csv',
    'культура':   DATA_DIR + '2гис_культура.csv',
    'образование':DATA_DIR + '2гис_образование.csv',
}
infra_df = processor.load_all(file_map)


  Загружаем: еда...
    → 309 объектов
  Загружаем: здоровье...
    → 222 объектов
  Загружаем: культура...
    → 93 объектов
  Загружаем: образование...
    → 611 объектов

Итого POI: 1235


### 2.2 Расчёт POI-признаков

In [7]:
def build_poi_features(apartments, poi, radii=[500, 1000]):
    mean_lat = apartments['Latitude'].mean()
    k_lat = 111320
    k_lon = 111320 * cos(radians(mean_lat))
    apt_coords = apartments[['Latitude', 'Longitude']].values * [k_lat, k_lon]
    poi_coords = poi[['Широта', 'Долгота']].values * [k_lat, k_lon]
    print(f'  Матрица расстояний: {len(apartments)} x {len(poi)}...')
    dist_matrix = cdist(apt_coords, poi_coords, metric='euclidean')
    categories  = poi['category'].values
    subtypes    = poi['subtype'].values
    unique_cats = poi['category'].unique()
    unique_subs = poi['subtype'].unique()
    features = pd.DataFrame(index=apartments.index)
    for radius in radii:
        in_radius = dist_matrix <= radius
        features[f'total_poi_{radius}m'] = in_radius.sum(axis=1)
        min_dist = np.where(in_radius, dist_matrix, np.inf).min(axis=1)
        features[f'min_distance_{radius}m'] = np.where(
            min_dist == np.inf, radius, min_dist).round(1)
        for cat in unique_cats:
            features[f'n_{cat}_{radius}m'] = in_radius[:, categories == cat].sum(axis=1)
        for sub in unique_subs:
            cnt = in_radius[:, subtypes == sub].sum(axis=1)
            if cnt.sum() > 0:
                features[f'n_{sub}_{radius}m'] = cnt
    return features

print('Создание POI-признаков...')
poi_features = build_poi_features(df_geo, infra_df, radii=POI_RADII)
df_poi = pd.concat([df_geo.reset_index(drop=True),
                    poi_features.reset_index(drop=True)], axis=1)
poi_cols = poi_features.columns.tolist()
df_poi[poi_cols] = df_poi[poi_cols].fillna(0)
print(f'Создано {poi_features.shape[1]} POI-признаков | Размер: {df_poi.shape}')
print(poi_features)


Создание POI-признаков...
  Матрица расстояний: 7143 x 1235...
Создано 38 POI-признаков | Размер: (7143, 53)
      total_poi_500m  min_distance_500m  n_еда_500m  n_здоровье_500m  \
0                  7              144.9           3                1   
1                  2              209.2           0                0   
2                  2               75.0           0                0   
3                  7              144.9           3                1   
4                  5              100.1           0                0   
...              ...                ...         ...              ...   
7138               7              155.8           3                0   
7139               3              161.2           0                0   
7140               3              206.0           1                0   
7141               0              500.0           0                0   
7142               0              500.0           0                0   

      n_культура_500m  n_о

## Этап 3. Детерминированные признаки + фильтр цены

Все преобразования ниже — детерминированные (без статистик по данным),
их безопасно применять до сплита.

### 3.1 Расстояние до центра и признаки этажности

`floor_ratio`, `is_first_floor`, `is_last_floor` — нелинейные взаимодействия,
которые XGBoost не всегда улавливает из отдельных `floor` и `floors_count`.


In [8]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

df_feat = df_poi.copy()

df_feat['distance_from_center_km'] = df_feat.apply(
    lambda r: round(haversine_km(r['Latitude'], r['Longitude'],
                                 CENTER_LAT, CENTER_LON), 3), axis=1)

df_feat['floor_ratio']    = (df_feat['floor'] / df_feat['floors_count']).round(3)
df_feat['is_first_floor'] = (df_feat['floor'] == 1).astype(int)
df_feat['is_last_floor']  = (df_feat['floor'] == df_feat['floors_count']).astype(int)

print(f'Размер после добавления признаков: {df_feat.shape}')
print(f'\nРасстояние до центра (км):')
print(df_feat['distance_from_center_km'].describe().round(3))


Размер после добавления признаков: (7143, 57)

Расстояние до центра (км):
count    7143.000
mean        5.979
std         4.153
min         0.252
25%         3.757
50%         5.779
75%         7.596
max        43.368
Name: distance_from_center_km, dtype: float64


### 3.2 Фильтр нерыночных сделок

In [9]:
df_feat['price_per_sqm_raw'] = (df_feat['price'] / df_feat['total_meters']).round()

before = len(df_feat)
df_feat = df_feat[df_feat['price_per_sqm_raw'] >= 20_000].reset_index(drop=True)
df_feat = df_feat.drop(columns=['price_per_sqm_raw'])

print(f'Удалено с ценой < 20 000 руб/м2: {before - len(df_feat)}')
print(f'Размер перед сплитом: {df_feat.shape}')


Удалено с ценой < 20 000 руб/м2: 4
Размер перед сплитом: (7139, 57)


## Этап 4. Разделение train / test

Сплит выполняется здесь — до любых статистических преобразований
(импутация, нормализация). Это предотвращает утечку данных.

Далее все статистики вычисляются только по `train_idx` и применяются к обеим выборкам.


In [10]:
train_idx, test_idx = train_test_split(
    df_feat.index, test_size=0.2, random_state=42
)
print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')
print(f'Test доля: {len(test_idx)/len(df_feat):.1%}')

y_price = df_feat['price']
y_log   = np.log(df_feat['price'])

y_train_log = y_log.loc[train_idx]
y_test_log  = y_log.loc[test_idx]
y_train     = y_price.loc[train_idx]
y_test      = y_price.loc[test_idx]

print(f'\nЦелевая переменная (log price):')
print(f'  Train: mean={y_train_log.mean():.3f}, std={y_train_log.std():.3f}')
print(f'  Test:  mean={y_test_log.mean():.3f},  std={y_test_log.std():.3f}')


Train: 5711 | Test: 1428
Test доля: 20.0%

Целевая переменная (log price):
  Train: mean=15.524, std=0.485
  Test:  mean=15.506,  std=0.475


## Этап 5. Импутация пропущенных значений

Все статистики вычисляются только по train. Применяются к полному датасету.

### 5.1 Флаги импутации

Создаются до заполнения пропусков — чтобы модель различала настоящие
и расчётные значения площади и года.


In [11]:
df_imp = df_feat.copy()

df_imp['living_imputed']  = df_imp['living_meters'].isna().astype(int)
df_imp['kitchen_imputed'] = df_imp['kitchen_meters'].isna().astype(int)
df_imp['year_imputed']    = df_imp['year_of_construction'].isna().astype(int)

print('Флаги импутации созданы:')
print(df_imp[['living_imputed','kitchen_imputed','year_imputed']].sum().to_string())


Флаги импутации созданы:
living_imputed     1661
kitchen_imputed     535
year_imputed       1053


### 5.2 `rooms_count` — мода по train

In [12]:
mode_rooms = df_imp.loc[train_idx, 'rooms_count'].mode().iloc[0]
df_imp['rooms_count'] = df_imp['rooms_count'].fillna(mode_rooms)
print(f'rooms_count: заполнено модой (из train) = {mode_rooms}')


rooms_count: заполнено модой (из train) = 2.0


### 5.3 `year_of_construction` — иерархическая импутация по train

Стратегия: адрес → улица → район → глобальная медиана.
Все уровни вычислены исключительно по train.


In [13]:
def fit_year_imputer(train_df: pd.DataFrame):
    """Вычисляет справочники для иерархической импутации по train."""
    train = train_df.copy()
    train['addr_key'] = (train['street'].astype(str).str.strip() + '|' +
                         train['house_number'].astype(str).str.strip())
    mode_by_addr = train.groupby('addr_key')['year_of_construction'].agg(
        lambda s: s.dropna().mode().iloc[0] if s.dropna().size > 0 else np.nan)
    med_street   = train.groupby('street')['year_of_construction'].median()
    med_district = train.groupby('district')['year_of_construction'].median()
    global_med   = train['year_of_construction'].median()
    return dict(mode_by_addr=mode_by_addr, med_street=med_street,
                med_district=med_district, global_med=global_med)


def apply_year_imputer(df: pd.DataFrame, refs: dict) -> pd.Series:
    """Применяет справочники к любому подмножеству данных."""
    df = df.copy()
    df['addr_key'] = (df['street'].astype(str).str.strip() + '|' +
                      df['house_number'].astype(str).str.strip())
    result = df['year_of_construction'].copy()
    result = result.fillna(df['addr_key'].map(refs['mode_by_addr']))
    result = result.fillna(df['street'].map(refs['med_street']))
    result = result.fillna(df['district'].map(refs['med_district']))
    result = result.fillna(refs['global_med'])
    return result.round().clip(1900, 2025).astype(int)


year_refs = fit_year_imputer(df_imp.loc[train_idx])

# Кросс-валидация качества импутации (только на train)
mask_known_train = df_imp.loc[train_idx, 'year_of_construction'].notna()
df_known = df_imp.loc[train_idx][mask_known_train].reset_index(drop=True)
mae_list, rmse_list = [], []
np.random.seed(42)
for _ in range(5):
    idx = df_known.sample(frac=0.2).index
    df_test_cv = df_known.copy()
    y_true = df_test_cv.loc[idx, 'year_of_construction'].copy()
    df_test_cv.loc[idx, 'year_of_construction'] = np.nan
    refs_cv = fit_year_imputer(df_test_cv.drop(index=idx))
    y_pred_cv = apply_year_imputer(df_test_cv.loc[idx], refs_cv)
    mae_list.append(mean_absolute_error(y_true, y_pred_cv))
    rmse_list.append(np.sqrt(mean_squared_error(y_true, y_pred_cv)))

print('Качество иерархической импутации (5-fold CV на train):')
print(f'   MAE:  {np.mean(mae_list):.2f} +/- {np.std(mae_list):.2f} лет')
print(f'   RMSE: {np.mean(rmse_list):.2f} +/- {np.std(rmse_list):.2f} лет')

df_imp['year_of_construction'] = apply_year_imputer(df_imp, year_refs)
print('\nyear_of_construction заполнен')


Качество иерархической импутации (5-fold CV на train):
   MAE:  5.56 +/- 0.31 лет
   RMSE: 12.19 +/- 0.71 лет

year_of_construction заполнен


### 5.4 `living_meters` и `kitchen_meters` — коэффициенты по train

$$\tilde{L}_i = \hat{\alpha}(d_i) \cdot T_i \quad \text{если } L_i = \text{NaN}$$

Коэффициенты $\hat{\alpha}$ и $\hat{\beta}$ считаются только по train.

> **Ограничение:** для части объявлений `living_meters` становится линейной функцией
> `total_meters`. Это создаёт мультиколлинеарность, влияние которой оценивается
> в подходе B2 (Этап 7.3).


In [14]:
train_data = df_imp.loc[train_idx]

nonnull_l = train_data.dropna(subset=['living_meters','total_meters'])
nonnull_k = train_data.dropna(subset=['kitchen_meters','total_meters'])

global_alpha = (nonnull_l['living_meters']  / nonnull_l['total_meters']).median()
global_beta  = (nonnull_k['kitchen_meters'] / nonnull_k['total_meters']).median()

alpha_by_district = nonnull_l.groupby('district').apply(
    lambda g: (g['living_meters'] / g['total_meters']).median(), include_groups=False)
beta_by_district  = nonnull_k.groupby('district').apply(
    lambda g: (g['kitchen_meters'] / g['total_meters']).median(), include_groups=False)

print(f'alpha_global (living/total):  {global_alpha:.3f}  [из train]')
print(f'beta_global  (kitchen/total): {global_beta:.3f}  [из train]')

def impute_ratio(row, col, ratio_by_district, global_ratio):
    if pd.isna(row[col]):
        ratio = ratio_by_district.get(row['district'], global_ratio)
        return ratio * row['total_meters']
    return row[col]

df_imp['living_meters']  = df_imp.apply(
    impute_ratio, axis=1, args=('living_meters',  alpha_by_district, global_alpha)).round(1)
df_imp['kitchen_meters'] = df_imp.apply(
    impute_ratio, axis=1, args=('kitchen_meters', beta_by_district,  global_beta)).round(1)

nulls = df_imp[['rooms_count','year_of_construction',
                'living_meters','kitchen_meters']].isnull().sum()
print('\nПропуски после импутации:')
print(nulls)


alpha_global (living/total):  0.604  [из train]
beta_global  (kitchen/total): 0.167  [из train]

Пропуски после импутации:
rooms_count             0
year_of_construction    0
living_meters           0
kitchen_meters          0
dtype: int64


In [15]:
df_imp['rooms_count']          = df_imp['rooms_count'].round().astype(int)
df_imp['total_meters']         = df_imp['total_meters'].round(1)
df_imp['living_meters']        = df_imp['living_meters'].round(1)
df_imp['kitchen_meters']       = df_imp['kitchen_meters'].round(1)
df_imp['year_of_construction'] = df_imp['year_of_construction'].astype(int)

print(f'Размер после импутации: {df_imp.shape}')
print(df_imp.head(3))


Размер после импутации: (7139, 60)
                                         url  floor  floors_count  \
0  https://omsk.cian.ru/sale/flat/314205396/     14            14   
1  https://omsk.cian.ru/sale/flat/312208029/      1             5   
2  https://omsk.cian.ru/sale/flat/314205398/     14            14   

   rooms_count  total_meters    price  year_of_construction  living_meters  \
0            1          29.5  3540000                  2023           17.7   
1            1          30.7  3350000                  1964           18.0   
2            1          32.8  3936000                  2023           19.7   

   kitchen_meters               district  ... n_детские_сады_1000m  \
0            20.1            Центральный  ...                    8   
1             7.0  мкр. Город Нефтяников  ...                    3   
2            19.5            Центральный  ...                    8   

  n_университеты_1000m n_колледжи_1000m  distance_from_center_km  floor_ratio  \
0            

## Этап 6. Выбросы и нормализация

Все границы вычисляются только по train и применяются к полному датасету.


In [16]:
df_final = df_imp.copy()
train_data = df_final.loc[train_idx]

# Winsorizing цены: только верхняя граница (99-й перцентиль по train)
# Нижняя граница через IQR может уйти в отрицательную — для цен это бессмысленно
price_upper = train_data['price'].quantile(0.99)
df_final['price'] = df_final['price'].clip(upper=price_upper)
print(f'price: обрезана сверху до {price_upper:,.0f} руб. [по train]')

# Обрезка физических признаков по 1-99 перцентилям train
clip_cols = ['total_meters', 'living_meters', 'kitchen_meters', 'floor', 'floors_count']
clip_bounds = {}
for col in clip_cols:
    p01 = train_data[col].quantile(0.01)
    p99 = train_data[col].quantile(0.99)
    clip_bounds[col] = (p01, p99)
    df_final[col] = df_final[col].clip(p01, p99)
    print(f'  {col}: [{p01:.1f}, {p99:.1f}]  [по train]')

# price_per_sqm — только для информации, не используется как признак
df_final['price_per_sqm'] = (df_final['price'] / df_final['total_meters']).round().astype(int)
df_final['price_per_sqm'] = df_final['price_per_sqm'].clip(
    upper=train_data.apply(lambda r: r['price'] / r['total_meters'], axis=1).quantile(0.99))

df_final['log_price'] = np.log(df_final['price'])

df_final.to_csv(OUTPUT_DIR + 'df_final.csv', index=False, encoding='utf-8')
print(f'\nИтоговый размер: {df_final.shape}')
print(f'Сохранено: df_final.csv')


price: обрезана сверху до 21,685,900 руб. [по train]
  total_meters: [21.2, 140.0]  [по train]
  living_meters: [12.0, 88.0]  [по train]
  kitchen_meters: [3.9, 30.0]  [по train]
  floor: [1.0, 15.0]  [по train]
  floors_count: [2.0, 19.0]  [по train]

Итоговый размер: (7139, 62)
Сохранено: df_final.csv


## Этап 7. Подготовка признаков

| Подход | `district` | `street` | `house_number` | Число фичей |
|--------|-----------|---------|---------------|------------|
| **A (OHE)** | OHE | OHE | OHE | ~1400 |
| **B (Target Enc.)** | OHE | Target Encoding | убран | ~120 |
| **B2** | OHE | Target Encoding | убран | ~118 (без living/kitchen) |


In [17]:
DROP_FOR_FEATURES = ['url', 'address', 'Latitude', 'Longitude',
                     'price', 'price_per_sqm', 'log_price']

X_all = df_final.drop(columns=[c for c in DROP_FOR_FEATURES if c in df_final.columns])

print(f'Train: {len(train_idx)}, Test: {len(test_idx)}')
print(f'Всего признаков до кодирования: {X_all.shape[1]}')

new_feats = ['floor_ratio','is_first_floor','is_last_floor',
             'living_imputed','kitchen_imputed','year_imputed']
print('\nНовые признаки:')
for f in new_feats:
    print(f'  {f}: {f in X_all.columns}  (среднее = {X_all[f].mean():.3f})')


Train: 5711, Test: 1428
Всего признаков до кодирования: 55

Новые признаки:
  floor_ratio: True  (среднее = 0.593)
  is_first_floor: True  (среднее = 0.142)
  is_last_floor: True  (среднее = 0.188)
  living_imputed: True  (среднее = 0.233)
  kitchen_imputed: True  (среднее = 0.075)
  year_imputed: True  (среднее = 0.147)


### 7.1 Подход A: OneHotEncoding

In [18]:
X_A = X_all.copy()
cat_cols_A = X_A.select_dtypes(include='object').columns.tolist()
num_cols_A = X_A.select_dtypes(exclude='object').columns.tolist()

preprocessor_A = ColumnTransformer([
    ('num', StandardScaler(), num_cols_A),
    ('cat', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), cat_cols_A)
])

X_A_train = preprocessor_A.fit_transform(X_A.loc[train_idx])
X_A_test  = preprocessor_A.transform(X_A.loc[test_idx])

cat_names_A = preprocessor_A.named_transformers_['cat']    .get_feature_names_out(cat_cols_A).tolist()
all_names_A = num_cols_A + cat_names_A

print(f'Подход A (OHE): {X_A_train.shape[1]} признаков')
print(f'  Числовых: {len(num_cols_A)}, Категориальных после OHE: {len(cat_names_A)}')


Подход A (OHE): 1397 признаков
  Числовых: 52, Категориальных после OHE: 1345


### 7.2 Подход B: Target Encoding для улицы

In [19]:
X_B = X_all.drop(columns=['house_number'], errors='ignore').copy()

# Target encoding считается только по train
train_with_price = X_B.loc[train_idx].copy()
train_with_price['price'] = y_price.loc[train_idx]
global_mean_price = train_with_price['price'].mean()
street_mean = train_with_price.groupby('street')['price'].mean()

X_B['street_target_encoded'] = X_B['street'].map(street_mean).fillna(global_mean_price)
X_B = X_B.drop(columns=['street'])

cat_cols_B = X_B.select_dtypes(include='object').columns.tolist()
num_cols_B = X_B.select_dtypes(exclude='object').columns.tolist()

preprocessor_B = ColumnTransformer([
    ('num', StandardScaler(), num_cols_B),
    ('cat', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), cat_cols_B)
])

X_B_train = preprocessor_B.fit_transform(X_B.loc[train_idx])
X_B_test  = preprocessor_B.transform(X_B.loc[test_idx])

cat_names_B = preprocessor_B.named_transformers_['cat']    .get_feature_names_out(cat_cols_B).tolist()
all_names_B = num_cols_B + cat_names_B

print(f'Подход B (Target Enc.): {X_B_train.shape[1]} признаков')
print(f'  Числовых: {len(num_cols_B)}, Категориальных после OHE: {len(cat_names_B)}')


Подход B (Target Enc.): 122 признаков
  Числовых: 53, Категориальных после OHE: 69


### 7.3 Подход B2: без `living_meters` и `kitchen_meters`

Используется для количественной оценки влияния мультиколлинеарности,
возникшей при импутации через `total_meters`.
Разница метрик B vs B2 приводится в разделе «Ограничения» диссертации.


In [20]:
cols_to_remove   = ['living_meters', 'kitchen_meters']
remaining_num_B2 = [c for c in num_cols_B if c not in cols_to_remove]

preprocessor_B2 = ColumnTransformer([
    ('num', StandardScaler(), remaining_num_B2),
    ('cat', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), cat_cols_B)
])

X_B2_train = preprocessor_B2.fit_transform(X_B.loc[train_idx])
X_B2_test  = preprocessor_B2.transform(X_B.loc[test_idx])

cat_names_B2 = preprocessor_B2.named_transformers_['cat']    .get_feature_names_out(cat_cols_B).tolist()
all_names_B2 = remaining_num_B2 + cat_names_B2

print(f'Подход B2 (без living/kitchen): {X_B2_train.shape[1]} признаков')
print(f'  Убраны: {cols_to_remove}')


Подход B2 (без living/kitchen): 120 признаков
  Убраны: ['living_meters', 'kitchen_meters']


## Этап 8. Обучение моделей

### 8.1 Scorers и пространство гиперпараметров

Два scorer-а возвращают MAE в рублях (в исходном масштабе):
- `log_mae_scorer` — для моделей, обученных на log(price)
- `raw_mae_scorer` — для моделей, обученных на сырой цене

Оба возвращают положительное значение MAE для корректного сравнения в сводной таблице.


In [21]:
def exp_mae_scorer(y_true_log, y_pred_log):
    return -mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))

log_mae_scorer = make_scorer(exp_mae_scorer, greater_is_better=False)
raw_mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

param_dist = {
    'n_estimators':     randint(200, 800),
    'max_depth':        randint(4, 14),
    'learning_rate':    uniform(0.01, 0.15),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
    'min_child_weight': randint(1, 10),
    'reg_alpha':        uniform(0, 1),
    'reg_lambda':       uniform(0, 2),
}

print('Scorers и пространство гиперпараметров определены')


Scorers и пространство гиперпараметров определены


### 8.2 Подбор гиперпараметров

Подбор выполняется отдельно для каждой конфигурации признаков (A, B, B2).
Это важно: оптимальная глубина дерева и число эстиматоров различаются
при 1400 признаках (A) и 120 признаках (B).

25 итераций x 3 фолда = 75 обучений на конфигурацию (~20 мин суммарно).


In [22]:
def tune_params(X_train, y_train, scorer, label, n_iter=25):
    print(f'Подбор: {label} ...')
    search = RandomizedSearchCV(
        XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=KFold(n_splits=3, shuffle=True, random_state=42),
        scoring=scorer,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X_train, y_train)
    params = search.best_params_
    params.update({'random_state': 42, 'verbosity': 0, 'n_jobs': -1})
    print(f'  n_estimators={params["n_estimators"]}, '
          f'max_depth={params["max_depth"]}, '
          f'lr={params["learning_rate"]:.4f}')
    return params

params_A  = tune_params(X_A_train,  y_train_log, log_mae_scorer, 'A  | log(price)')
params_B  = tune_params(X_B_train,  y_train_log, log_mae_scorer, 'B  | log(price)')
params_B2 = tune_params(X_B2_train, y_train_log, log_mae_scorer, 'B2 | log(price)')


Подбор: A  | log(price) ...
  n_estimators=671, max_depth=11, lr=0.1580
Подбор: B  | log(price) ...
  n_estimators=671, max_depth=11, lr=0.1580
Подбор: B2 | log(price) ...
  n_estimators=671, max_depth=11, lr=0.1580


### 8.3 Функции обучения

`evaluate_log` — обучение на log(price), предсказание через exp().
`evaluate_raw` — обучение на сырой цене.

Все метрики в обоих случаях выводятся в рублях.
CV MAE всегда положителен — для корректного сравнения между функциями.


In [23]:
def _print_metrics(m):
    print(f"  {m['Подход']}  [{m['Таргет']}]")
    print(f"{'─'*58}")
    print(f"  R2:         {m['Test R2']:.4f}")
    print(f"  MAE:        {m['Test MAE']:,.0f} руб.")
    print(f"  Median AE:  {m['Median AE']:,.0f} руб.")
    print(f"  RMSE:       {m['Test RMSE']:,.0f} руб.")
    print(f"  MAPE:       {m['MAPE, %']:.2f}%")
    print(f"  CV MAE:     {m['CV MAE']:,.0f} +/- {m['CV MAE std']:,.0f} руб.")
    print(f"  Признаков:  {m['N features']}")


def evaluate_log(X_train, X_test, y_train_log, y_test, name, params):
    """Обучение на log(price), предсказание через exp()."""
    model = XGBRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train_log,
                                cv=cv, scoring=log_mae_scorer, n_jobs=-1)
    model.fit(X_train, y_train_log)
    y_pred = np.exp(model.predict(X_test))
    m = {
        'Подход':     name,
        'Таргет':     'log(price)',
        'CV MAE':     np.mean(cv_scores),    # положительное (двойная инверсия scorer)
        'CV MAE std': np.std(cv_scores),
        'Test MAE':   mean_absolute_error(y_test, y_pred),
        'Median AE':  median_absolute_error(y_test, y_pred),
        'Test RMSE':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'Test R2':    r2_score(y_test, y_pred),
        'MAPE, %':    np.mean(np.abs((y_test - y_pred) / y_test)) * 100,
        'N features': X_train.shape[1]
    }
    _print_metrics(m)
    return m, model


def evaluate_raw(X_train, X_test, y_train, y_test, name, params):
    """Обучение на сырой цене."""
    model = XGBRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=cv, scoring=raw_mae_scorer, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    m = {
        'Подход':     name,
        'Таргет':     'price (raw)',
        'CV MAE':     -np.mean(cv_scores),   # raw_scorer отрицательный — инвертируем
        'CV MAE std': np.std(cv_scores),
        'Test MAE':   mean_absolute_error(y_test, y_pred),
        'Median AE':  median_absolute_error(y_test, y_pred),
        'Test RMSE':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'Test R2':    r2_score(y_test, y_pred),
        'MAPE, %':    np.mean(np.abs((y_test - y_pred) / y_test)) * 100,
        'N features': X_train.shape[1]
    }
    _print_metrics(m)
    return m, model


print('Функции обучения определены')


Функции обучения определены


### 8.4 Обучение всех моделей

In [24]:
print('Обучаем модели...\n')

met_A_log,  mod_A_log  = evaluate_log(X_A_train,  X_A_test,  y_train_log, y_test,
                                       'A: OHE', params_A) # 30 сек
met_A_raw,  mod_A_raw  = evaluate_raw(X_A_train,  X_A_test,  y_train,     y_test,
                                       'A: OHE', params_A) # 20 cек
met_B_log,  mod_B_log  = evaluate_log(X_B_train,  X_B_test,  y_train_log, y_test,
                                       'B: Target Enc', params_B) #10 сек
met_B_raw,  mod_B_raw  = evaluate_raw(X_B_train,  X_B_test,  y_train,     y_test,
                                       'B: Target Enc', params_B) #13 сек
met_B2_log, mod_B2_log = evaluate_log(X_B2_train, X_B2_test, y_train_log, y_test,
                                       'B2: Target Enc (без living/kitchen)', params_B2) #13 сек
met_B2_raw, mod_B2_raw = evaluate_raw(X_B2_train, X_B2_test, y_train,     y_test,
                                       'B2: Target Enc (без living/kitchen)', params_B2) #10 сек

all_metrics = [met_A_log, met_A_raw, met_B_log, met_B_raw, met_B2_log, met_B2_raw]
all_models  = [mod_A_log, mod_A_raw, mod_B_log, mod_B_raw, mod_B2_log, mod_B2_raw]
all_X_test  = [X_A_test,  X_A_test,  X_B_test,  X_B_test,  X_B2_test, X_B2_test]
all_is_log  = [True,      False,     True,      False,     True,       False]


Обучаем модели...

  A: OHE  [log(price)]
──────────────────────────────────────────────────────────
  R2:         0.8977
  MAE:        673,438 руб.
  Median AE:  414,300 руб.
  RMSE:       1,122,108 руб.
  MAPE:       11.18%
  CV MAE:     731,218 +/- 26,493 руб.
  Признаков:  1397
  A: OHE  [price (raw)]
──────────────────────────────────────────────────────────
  R2:         0.8920
  MAE:        691,283 руб.
  Median AE:  426,687 руб.
  RMSE:       1,152,710 руб.
  MAPE:       11.54%
  CV MAE:     737,073 +/- 23,350 руб.
  Признаков:  1397
  B: Target Enc  [log(price)]
──────────────────────────────────────────────────────────
  R2:         0.8924
  MAE:        687,517 руб.
  Median AE:  419,123 руб.
  RMSE:       1,150,558 руб.
  MAPE:       11.35%
  CV MAE:     742,558 +/- 31,959 руб.
  Признаков:  122
  B: Target Enc  [price (raw)]
──────────────────────────────────────────────────────────
  R2:         0.8915
  MAE:        698,944 руб.
  Median AE:  447,234 руб.
  RMSE:       1,1

In [25]:
print(mod_B_log.get_booster().feature_names)

None


### 8.5 Сравнительная таблица и выбор лучшей модели

In [26]:
comparison = pd.DataFrame(all_metrics).round({
    'CV MAE': 0, 'CV MAE std': 0, 'Test MAE': 0, 'Median AE': 0,
    'Test RMSE': 0, 'Test R2': 4, 'MAPE, %': 2
})

print('Сравнение всех моделей:')
print(comparison[['Подход','Таргет','CV MAE','Test MAE',
                  'Median AE','MAPE, %','Test R2','N features']].to_string(index=False))

# Влияние мультиколлинеарности (B vs B2, одинаковый таргет)
delta_mape_log = met_B_log['MAPE, %'] - met_B2_log['MAPE, %']
delta_mae_log  = met_B_log['Test MAE'] - met_B2_log['Test MAE']
delta_mape_raw = met_B_raw['MAPE, %'] - met_B2_raw['MAPE, %']
print(f'\nВлияние мультиколлинеарности (B vs B2):')
print(f'  log: DELTA MAPE = {delta_mape_log:+.2f} п.п.,  DELTA MAE = {delta_mae_log:+,.0f} руб.')
print(f'  raw: DELTA MAPE = {delta_mape_raw:+.2f} п.п.')
print(f'  (>0 означает: living/kitchen улучшают качество, несмотря на мультиколлинеарность)')

comparison.to_csv(OUTPUT_DIR + 'model_comparison_full.csv', index=False)
print(f'\nСохранено: model_comparison_full.csv')

# Выбор лучшей модели по CV MAE (все значения положительные — берём idxmin)
best_idx    = comparison['CV MAE'].idxmin()
best_row    = comparison.loc[best_idx]
best_model  = all_models[best_idx]
best_X_test = all_X_test[best_idx]
best_is_log = all_is_log[best_idx]

print(f'\nЛучший подход (по CV MAE): {best_row["Подход"]}  [{best_row["Таргет"]}]')
print(f'  CV MAE   = {best_row["CV MAE"]:,.0f} руб.')
print(f'  Test MAE = {best_row["Test MAE"]:,.0f} руб.')
print(f'  MAPE     = {best_row["MAPE, %"]:.2f}%')
print(f'  R2       = {best_row["Test R2"]:.4f}')


Сравнение всех моделей:
                             Подход      Таргет   CV MAE  Test MAE  Median AE  MAPE, %  Test R2  N features
                             A: OHE  log(price) 731218.0  673438.0   414300.0    11.18   0.8977        1397
                             A: OHE price (raw) 737073.0  691283.0   426687.0    11.54   0.8920        1397
                      B: Target Enc  log(price) 742558.0  687517.0   419123.0    11.35   0.8924         122
                      B: Target Enc price (raw) 744737.0  698944.0   447234.0    11.79   0.8915         122
B2: Target Enc (без living/kitchen)  log(price) 767542.0  704942.0   414416.0    11.43   0.8824         120
B2: Target Enc (без living/kitchen) price (raw) 784261.0  726385.0   442598.0    12.31   0.8857         120

Влияние мультиколлинеарности (B vs B2):
  log: DELTA MAPE = -0.09 п.п.,  DELTA MAE = -17,425 руб.
  raw: DELTA MAPE = -0.53 п.п.
  (>0 означает: living/kitchen улучшают качество, несмотря на мультиколлинеарность)

Сохра

### 8.6 Анализ ошибок по ценовым сегментам

Единый MAPE скрывает разницу между дешёвым и дорогим жильём.
Таблица ниже сохраняется в CSV для вставки в диссертацию.


In [27]:
if best_is_log:
    y_pred_best = np.exp(best_model.predict(best_X_test))
else:
    y_pred_best = best_model.predict(best_X_test)

test_df = df_final.loc[test_idx].copy()
test_df['predicted_price'] = y_pred_best
test_df['abs_error'] = np.abs(test_df['price'] - test_df['predicted_price'])
test_df['error_pct'] = test_df['abs_error'] / test_df['price'] * 100

bins   = [0, 2_500_000, 4_000_000, 6_000_000, 10_000_000, np.inf]
labels = ['до 2.5 млн', '2.5-4 млн', '4-6 млн', '6-10 млн', 'свыше 10 млн']
test_df['price_bin'] = pd.cut(test_df['price'], bins=bins, labels=labels)

segment_table = test_df.groupby('price_bin', observed=True).agg(
    Объектов=('price', 'size'),
    Средняя_цена_млн=('price', lambda x: round(x.mean() / 1e6, 2)),
    MAE_тыс=('abs_error', lambda x: round(x.mean() / 1e3, 1)),
    Median_AE_тыс=('abs_error', lambda x: round(x.median() / 1e3, 1)),
    MAPE_pct=('error_pct', lambda x: round(x.mean(), 2))
).rename(columns={'MAPE_pct': 'MAPE, %'})

print(segment_table.to_string())

segment_table.to_csv(OUTPUT_DIR + 'segment_errors_table.csv', encoding='utf-8-sig')
print(f'\nСохранено: segment_errors_table.csv')


              Объектов  Средняя_цена_млн  MAE_тыс  Median_AE_тыс  MAPE, %
price_bin                                                                
до 2.5 млн          45              1.97    669.4          455.8    35.85
2.5-4 млн          328              3.39    364.4          273.6    10.85
4-6 млн            539              4.97    476.6          358.5     9.60
6-10 млн           374              7.54    775.9          575.3    10.25
свыше 10 млн       142             14.24   1885.1         1437.4    12.81

Сохранено: segment_errors_table.csv


### 8.7 Важность признаков (XGBoost feature importance)

In [28]:
imp_A = pd.DataFrame({'feature': all_names_A,
                      'importance': mod_A_log.feature_importances_})          .sort_values('importance', ascending=False)

imp_B = pd.DataFrame({'feature': all_names_B,
                      'importance': mod_B_log.feature_importances_})          .sort_values('importance', ascending=False)

print('Топ-20 признаков (Подход A — OHE):')
print(imp_A.head(20).to_string(index=False))
print('\nТоп-20 признаков (Подход B — Target Encoding):')
print(imp_B.head(20).to_string(index=False))

new_feats = ['floor_ratio','is_first_floor','is_last_floor',
             'living_imputed','kitchen_imputed','year_imputed']
print('\nВажность новых признаков (Подход B):')
for f in new_feats:
    row = imp_B[imp_B['feature'] == f]
    if not row.empty:
        rank = imp_B.index.get_loc(row.index[0]) + 1
        print(f'  {f}: importance={row.iloc[0]["importance"]:.4f} (ранг {rank})')
    else:
        print(f'  {f}: не найден')

imp_A.to_csv(OUTPUT_DIR + 'feature_importance_A.csv', index=False)
imp_B.to_csv(OUTPUT_DIR + 'feature_importance_B.csv', index=False)
print('\nСохранено: feature_importance_A/B.csv')


Топ-20 признаков (Подход A — OHE):
                  feature  importance
street_ 1-я Автомобильная    0.050701
district_Крутая Горка мкр    0.040919
      street_ 1-я Осенняя    0.025901
        street_Граничная     0.020354
 street_проспект Культуры    0.018732
         house_number_110    0.015817
             total_meters    0.015566
     street_ 2-я Учхозная    0.014184
       house_number_165/2    0.014045
       street_ Пархоменко    0.013023
       district_Советский    0.012337
   street_ Маршала Жукова    0.011398
         house_number_186    0.010986
        house_number_44к5    0.010016
          house_number_64    0.009838
     street_ Осташковская    0.009752
    street_ Б.Г. Шаронова    0.009575
      street_ Маяковского    0.009290
            living_meters    0.008901
     street_ Красный Путь    0.008382

Топ-20 признаков (Подход B — Target Encoding):
                     feature  importance
       street_target_encoded    0.094076
                total_meters    0.090

### 8.8 SHAP-анализ

SHAP даёт интерпретацию на уровне отдельных наблюдений.
Используется подход B (меньше признаков — графики читаемее).

Сохраняемые файлы:
- `shap_summary_plot.png` — beeswarm, показывает направление и силу влияния
- `shap_bar_plot.png` — средний |SHAP|, аналог feature importance
- `shap_dep_*.png` — dependence plots для топ-3 признаков
- `shap_waterfall_example.png` — объяснение одного конкретного предсказания


In [29]:
shap_model  = mod_B_log
shap_X_test = X_B_test
shap_names  = all_names_B

print('Вычисляем SHAP-значения...')
explainer   = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(shap_X_test)

shap_importance = pd.DataFrame({
    'feature':   shap_names,
    'mean_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False)

print('\nГлобальная важность признаков (mean |SHAP|) — топ-20:')
print(shap_importance.head(20).to_string(index=False))
shap_importance.to_csv(OUTPUT_DIR + 'shap_importance.csv', index=False)
print('\nСохранено: shap_importance.csv')


Вычисляем SHAP-значения...

Глобальная важность признаков (mean |SHAP|) — топ-20:
                feature  mean_shap
           total_meters   0.150784
  street_target_encoded   0.085160
          living_meters   0.062033
         kitchen_meters   0.059877
           floors_count   0.036860
   year_of_construction   0.036665
            rooms_count   0.033303
distance_from_center_km   0.014723
        total_poi_1000m   0.010782
            n_еда_1000m   0.007577
            floor_ratio   0.007177
                  floor   0.006952
    n_образование_1000m   0.005000
      min_distance_500m   0.004739
      n_медцентры_1000m   0.003784
     min_distance_1000m   0.003625
         total_poi_500m   0.003543
       n_здоровье_1000m   0.003417
       n_рестораны_500m   0.003009
   n_детские_сады_1000m   0.002996

Сохранено: shap_importance.csv


In [30]:
# Summary plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_X_test,
                  feature_names=shap_names, max_display=20, show=False)
plt.title('SHAP Summary Plot — топ-20 признаков', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'shap_summary_plot.png', dpi=180, bbox_inches='tight')
plt.close()
print('Сохранено: shap_summary_plot.png')

# Bar plot
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, shap_X_test,
                  feature_names=shap_names, plot_type='bar',
                  max_display=20, show=False)
plt.title('Средний |SHAP| — топ-20 признаков', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'shap_bar_plot.png', dpi=180, bbox_inches='tight')
plt.close()
print('Сохранено: shap_bar_plot.png')


Сохранено: shap_summary_plot.png
Сохранено: shap_bar_plot.png


In [31]:
# Dependence plots для топ-3 признаков
top3 = shap_importance['feature'].iloc[:3].tolist()
for feat in top3:
    if feat in shap_names:
        feat_idx = shap_names.index(feat)
        plt.figure(figsize=(7, 5))
        shap.dependence_plot(feat_idx, shap_values, shap_X_test,
                             feature_names=shap_names, show=False)
        plt.title(f'SHAP dependence: {feat}', fontsize=12)
        plt.tight_layout()
        fname = f'shap_dep_{feat[:30].replace("/","_")}.png'
        plt.savefig(OUTPUT_DIR + fname, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Сохранено: {fname}')

# Waterfall — пример объяснения одного предсказания
sample_idx  = 0
explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=shap_X_test[sample_idx],
    feature_names=shap_names
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(explanation, max_display=15, show=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.close()

real_price = y_test.iloc[sample_idx]
pred_price = np.exp(shap_model.predict(shap_X_test[[sample_idx]]))[0]
print(f'\nПример объяснения (waterfall):')
print(f'  Реальная цена:      {real_price:,.0f} руб.')
print(f'  Предсказанная цена: {pred_price:,.0f} руб.')
print(f'  Ошибка:             {abs(real_price-pred_price):,.0f} руб. '
      f'({abs(real_price-pred_price)/real_price*100:.1f}%)')
print(f'Сохранено: shap_waterfall_example.png')


Сохранено: shap_dep_total_meters.png
Сохранено: shap_dep_street_target_encoded.png
Сохранено: shap_dep_living_meters.png

Пример объяснения (waterfall):
  Реальная цена:      4,449,000 руб.
  Предсказанная цена: 5,059,181 руб.
  Ошибка:             610,181 руб. (13.7%)
Сохранено: shap_waterfall_example.png


## Этап 9. Сохранение артефактов и предсказание

In [32]:
# Модели
joblib.dump(mod_A_log,      OUTPUT_DIR + 'model_A_log.pkl')
joblib.dump(mod_A_raw,      OUTPUT_DIR + 'model_A_raw.pkl')
joblib.dump(mod_B_log,      OUTPUT_DIR + 'model_B_log.pkl')
joblib.dump(mod_B_raw,      OUTPUT_DIR + 'model_B_raw.pkl')
joblib.dump(mod_B2_log,     OUTPUT_DIR + 'model_B2_log.pkl')
joblib.dump(mod_B2_raw,     OUTPUT_DIR + 'model_B2_raw.pkl')

# Препроцессоры
joblib.dump(preprocessor_A,  OUTPUT_DIR + 'preprocessor_A.pkl')
joblib.dump(preprocessor_B,  OUTPUT_DIR + 'preprocessor_B.pkl')
joblib.dump(preprocessor_B2, OUTPUT_DIR + 'preprocessor_B2.pkl')

# Target encoding
joblib.dump(street_mean,       OUTPUT_DIR + 'street_target_encoding.pkl')
joblib.dump(global_mean_price, OUTPUT_DIR + 'global_mean_price.pkl')

# Справочники импутации
joblib.dump(year_refs,         OUTPUT_DIR + 'year_imputer_refs.pkl')
joblib.dump(alpha_by_district, OUTPUT_DIR + 'alpha_by_district.pkl')
joblib.dump(beta_by_district,  OUTPUT_DIR + 'beta_by_district.pkl')
joblib.dump(global_alpha,      OUTPUT_DIR + 'global_alpha.pkl')
joblib.dump(global_beta,       OUTPUT_DIR + 'global_beta.pkl')
joblib.dump(clip_bounds,       OUTPUT_DIR + 'clip_bounds.pkl')

print(f'Все артефакты сохранены в {OUTPUT_DIR}')


Все артефакты сохранены в /content/output/


### 9.1 Предсказание на новых данных

In [33]:
def predict_prices(new_df: pd.DataFrame,
                   approach: str = 'B',
                   use_log: bool = True) -> pd.DataFrame:
    """
    Предсказание цен на новых объявлениях.

    new_df   : DataFrame
    approach : 'A' — OHE, 'B' — Target Encoding, 'B2' — без living/kitchen
    use_log  : True — модель обучена на log(price), False — на сырой цене
    """
    result = new_df.copy()

    model_key = f'model_{approach}_{"log" if use_log else "raw"}'
    model = joblib.load(OUTPUT_DIR + f'{model_key}.pkl')
    prep  = joblib.load(OUTPUT_DIR + f'preprocessor_{approach}.pkl')

    X_new = result.drop(columns=[c for c in DROP_FOR_FEATURES if c in result.columns])

    if approach in ('B', 'B2'):
        street_enc  = joblib.load(OUTPUT_DIR + 'street_target_encoding.pkl')
        global_mean = joblib.load(OUTPUT_DIR + 'global_mean_price.pkl')
        X_new = X_new.drop(columns=['house_number'], errors='ignore')
        X_new['street_target_encoded'] = X_new['street'].map(street_enc).fillna(global_mean)
        X_new = X_new.drop(columns=['street'])

    if approach == 'B2':
        X_new = X_new.drop(columns=['living_meters','kitchen_meters'], errors='ignore')
    print(list(X_new.columns))
    X_proc = prep.transform(X_new)
    y_pred = model.predict(X_proc)

    if use_log:
        y_pred = np.exp(y_pred)

    result['predicted_price'] = y_pred.round(0).astype(int)
    return result


# Пример: предсказание на тестовой выборке лучшей моделью
approach_map = {0: 'A', 1: 'A', 2: 'B', 3: 'B', 4: 'B2', 5: 'B2'}
best_approach = approach_map[best_idx]
test_preds = predict_prices(df_final.loc[test_idx].copy(),
                             approach=best_approach,
                             use_log=best_is_log)

mae_check  = mean_absolute_error(test_preds['price'], test_preds['predicted_price'])
med_check  = median_absolute_error(test_preds['price'], test_preds['predicted_price'])
mape_check = (np.abs(test_preds['price'] - test_preds['predicted_price'])
              / test_preds['price']).mean() * 100

print(f'Метрики на тестовой выборке (подход {best_approach}, log={best_is_log}):')
print(f'  MAE:       {mae_check:,.0f} руб.')
print(f'  Median AE: {med_check:,.0f} руб.')
print(f'  MAPE:      {mape_check:.2f}%')
print()
print('Пример предсказаний (первые 10 строк):')
print(test_preds[['address','rooms_count','total_meters',
                  'price','predicted_price']].head(10).to_string(index=False))

test_preds.to_csv(OUTPUT_DIR + 'test_predictions.csv', index=False, encoding='utf-8')
print(f'\nСохранено: test_predictions.csv')


['floor', 'floors_count', 'rooms_count', 'total_meters', 'year_of_construction', 'living_meters', 'kitchen_meters', 'district', 'street', 'house_number', 'total_poi_500m', 'min_distance_500m', 'n_еда_500m', 'n_здоровье_500m', 'n_культура_500m', 'n_образование_500m', 'n_кафе_500m', 'n_рестораны_500m', 'n_медцентры_500m', 'n_больницы_500m', 'n_кинотеатры_500m', 'n_музеи_500m', 'n_театры_500m', 'n_гимназии_500m', 'n_лицеи_500m', 'n_школы_500m', 'n_детские_сады_500m', 'n_университеты_500m', 'n_колледжи_500m', 'total_poi_1000m', 'min_distance_1000m', 'n_еда_1000m', 'n_здоровье_1000m', 'n_культура_1000m', 'n_образование_1000m', 'n_кафе_1000m', 'n_рестораны_1000m', 'n_медцентры_1000m', 'n_больницы_1000m', 'n_кинотеатры_1000m', 'n_музеи_1000m', 'n_театры_1000m', 'n_гимназии_1000m', 'n_лицеи_1000m', 'n_школы_1000m', 'n_детские_сады_1000m', 'n_университеты_1000m', 'n_колледжи_1000m', 'distance_from_center_km', 'floor_ratio', 'is_first_floor', 'is_last_floor', 'living_imputed', 'kitchen_imputed',

In [34]:
print(mod_B_log.get_booster().feature_names)

None


# Линейная регрессия

In [35]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

# 1. Чистая линейная регрессия
lr = LinearRegression()
lr.fit(X_B_train, y_train_log)
y_pred_lr = np.exp(lr.predict(X_B_test))

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
med_lr  = median_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)
mape_lr = np.mean(np.abs((y_test - y_pred_lr) / y_test)) * 100

print(f'Линейная регрессия (B, log(price)):')
print(f'  MAE:       {mae_lr:,.0f} руб.')
print(f'  Median AE: {med_lr:,.0f} руб.')
print(f'  RMSE:      {rmse_lr:,.0f} руб.')
print(f'  R²:        {r2_lr:.4f}')
print(f'  MAPE:      {mape_lr:.2f}%')

# 2. Ridge — линейная регрессия с L2-регуляризацией
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_B_train, y_train_log)
y_pred_ridge = np.exp(ridge.predict(X_B_test))

mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)
mape_ridge = np.mean(np.abs((y_test - y_pred_ridge) / y_test)) * 100
r2_ridge   = r2_score(y_test, y_pred_ridge)

print(f'\nRidge (α=1.0, B, log(price)):')
print(f'  MAE:  {mae_ridge:,.0f} руб.')
print(f'  R²:   {r2_ridge:.4f}')
print(f'  MAPE: {mape_ridge:.2f}%')

Линейная регрессия (B, log(price)):
  MAE:       801,438 руб.
  Median AE: 472,696 руб.
  RMSE:      1,405,830 руб.
  R²:        0.8394
  MAPE:      12.87%

Ridge (α=1.0, B, log(price)):
  MAE:  802,590 руб.
  R²:   0.8386
  MAPE: 12.88%


In [36]:
import matplotlib.pyplot as plt

# 1. Scatter: предсказанные vs реальные
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_best, alpha=0.3, s=8)
max_val = max(y_test.max(), y_pred_best.max())
ax.plot([0, max_val], [0, max_val], 'r--', lw=1, label='Идеальный прогноз')
ax.set_xlabel('Реальная цена, руб.')
ax.set_ylabel('Предсказанная цена, руб.')
ax.set_title('Predicted vs Actual: лучшая модель (A log)')
ax.legend()
plt.savefig(OUTPUT_DIR + 'scatter_pred_vs_actual.png', dpi=150, bbox_inches='tight')

# 2. Гистограмма относительных остатков
residuals_pct = (y_test - y_pred_best) / y_test * 100
plt.figure(figsize=(9, 5))
plt.hist(residuals_pct, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', lw=1)
plt.xlabel('Относительная ошибка, %')
plt.ylabel('Число объектов')
plt.title('Распределение остатков (test)')
plt.savefig(OUTPUT_DIR + 'residuals_histogram.png', dpi=150, bbox_inches='tight')

In [37]:
import joblib
joblib.dump({
    "mode_rooms": int(mode_rooms),
    "year_refs": year_refs,
    "alpha_by_district": alpha_by_district,
    "beta_by_district": beta_by_district,
    "global_alpha": float(global_alpha),
    "global_beta": float(global_beta),
}, OUTPUT_DIR + "imputation_refs.pkl")

['/content/output/imputation_refs.pkl']

In [38]:
import sklearn, xgboost
print(sklearn.__version__, xgboost.__version__)

1.6.1 3.2.0


---

## Итоги

| Проблема | Решение |
|----------|---------|
| Утечка: импутация до сплита | Train/test split перенесён в Этап 4 |
| Утечка: winsorizing до сплита | Границы вычислены только по train (Этап 6) |
| Отрицательная нижняя граница цены | Winsorizing только сверху (99-й перцентиль) |
| Обучение на сырой цене | Добавлены модели log и raw, сравниваются явно |
| Выбор модели по R² | Выбор по CV MAE (Этап 8.5) |
| Знак CV MAE зависел от scorer | Оба scorer возвращают положительный MAE |
| Гиперпараметры только для B | Отдельный подбор для A, B, B2 (Этап 8.2) |
| Нет флагов импутации | Добавлены `living_imputed`, `kitchen_imputed`, `year_imputed` |
| Нет признаков этажности | Добавлены `floor_ratio`, `is_first_floor`, `is_last_floor` |
| Пропадали адреса при merge | Адрес формируется без strip() — как в coordinates.csv |
| Нет оценки мультиколлинеарности | Подход B2, DELTA MAPE выводится явно (Этап 8.5) |
| Нет Median AE | Добавлен во все сводки |
| Нет анализа по сегментам | Таблица сегментов (Этап 8.6) |
| Нет SHAP | Добавлен полный SHAP-анализ (Этап 8.8) |

**Файлы для диссертации:**

| Файл | Назначение |
|------|-----------|
| `model_comparison_full.csv` | Сравнение всех моделей |
| `segment_errors_table.csv` | Таблица ошибок по сегментам (Глава 4) |
| `shap_summary_plot.png` | SHAP beeswarm (Практическая значимость) |
| `shap_bar_plot.png` | SHAP bar plot |
| `shap_dep_*.png` | Dependence plots топ-3 признаков |
| `shap_waterfall_example.png` | Пример объяснения предсказания |
| `shap_importance.csv` | SHAP-важность всех признаков |
| `feature_importance_A/B.csv` | XGBoost feature importance |
| `test_predictions.csv` | Предсказания на тестовой выборке |
